In [ ]:
import pandas as pd
from nn.model_classes import Model, Layer, PolyLayer
from nn.functions import MSE, leaky_relu, der_leaky_relu, sigmoid, der_sigmoid, mirror, der_mirror
from nn.trainer import RegressionTrainer
from nn.plotter import Plotter
from nn.dataset_utils import pca, standardize_data, split_data
from nn.optimizers import SGD, Momentum

In [ ]:
dataset = pd.read_csv(r'..\expenses.csv')

In [ ]:
dataset.info()

In [ ]:
dataset.head()

In [ ]:
dataset = pd.get_dummies(dataset, drop_first = True, dtype = int)
dataset.head()

In [ ]:
dataset.info()

In [ ]:
D_train, D_test = split_data(dataset, 0.3, 10)

In [ ]:
D_train.info()

In [ ]:
D_test.info()

In [ ]:
X_train = D_train.drop(columns = ['charges'])
y_train = D_train['charges']

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
model = Model(MSE(), 5)
model.add_layer(Layer(8))
model.add_layer(PolyLayer(Layer(4, leaky_relu(), der_leaky_relu()), Layer(4, sigmoid, der_sigmoid)))
model.add_layer(PolyLayer(Layer(2, leaky_relu(0.01, 0.2), der_leaky_relu(0.01, 0.2)), Layer(2, leaky_relu(0.2, 0.01), der_leaky_relu(0.2, 0.01))))
model.add_layer(Layer(1, mirror, der_mirror))
model.compile()

In [ ]:
trainer = RegressionTrainer(model, SGD())

In [ ]:
X_train = X_train.to_numpy()
y_train = y_train.to_numpy()
y_train = y_train.reshape((y_train.shape[0], 1))
X_means, X_stds = standardize_data(X_train, [0, 1])
y_means, y_stds = standardize_data(y_train)

In [ ]:
trainer.train(X_train, y_train, 4, 0.02, 25)
#model.load_weights(r"models\dataset_regression.npz")

In [ ]:
trainer.save_history('./logs', 'dataset_regression')
#model.save_weights("./models", "dataset_regression")

In [ ]:
plotter = Plotter()
plotter.read_file(r'logs\dataset_regression.txt')
plotter.plot_gradients("./plots", "dataset_regression", 700)
plotter.plot_weights("./plots", "dataset_regression", 700)
plotter.plot_score("./plots", "dataset_regression", 700, False)

In [ ]:
X_test = D_test.drop(columns = ['charges'])
y_test = D_test['charges']

In [ ]:
X_test = X_test.to_numpy()
y_test = y_test.to_numpy()
y_test = y_test.reshape((y_test.shape[0], 1))

In [ ]:
standardize_data(X_test, [0, 1], from_means = X_means, from_stds = X_stds)
standardize_data(y_test, from_means = y_means, from_stds = y_stds)

In [ ]:
_ = trainer.predict(X_test, y_test)

In [ ]:
axis = pca(D_train, "charges", 1)
plotter.plot_regression(axis, D_train["charges"].to_numpy(), "./plots", "dataset_regression", x_label = "PCA Component-1")

In [ ]:
plotter.plot_contours(trainer, X_train, y_train, "./plots", "dataset_regression", magnitude = 0.2)